In [15]:
from langchain_gigachat.chat_models import GigaChat
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
import os 
from dotenv import load_dotenv

load_dotenv()

# Настройка модели GigaChat с расширенными параметрами
llm = GigaChat(
    credentials=os.getenv('GIGA_KEY'),
    model='GigaChat-2', 
    verify_ssl_certs=False,
    temperature=0.2, # настройка креативности
    max_tokens=1000 # максимальная длина ответа
)

# Проверка подключения
response = llm.invoke('Привет! Как дела?')
print(response.content)

Привет! Всё отлично, готов общаться и помогать. А у тебя как настроение сегодня?


In [16]:
# Шаблон системного сообщения
system_prompt = SystemMessagePromptTemplate.from_template(
    'Ты - эксперт по анализу текстовых заявок на аренду жилья.'
)

# Шаблон пользовательского сообщения
user_prompt = HumanMessagePromptTemplate.from_template(
    'Заявка: {text}\n\nИзвлеки количество человек и верни только число.'
)

# Собираем чат-шаблон
chat_prompt = ChatPromptTemplate.from_messages([system_prompt, user_prompt])

# При вызове цепочки:
chain = chat_prompt | llm | StrOutputParser()
result = chain.invoke({'text': 'Семья из трёх человек снимает квартиру'})
print(result) # ожидаем 3

3


In [17]:
# Простой промпт для извлечения количества людей
basic_prompt = PromptTemplate(
    input_variables=['text'],
    template='''
Проанализируй следующий текст заявки на аренду жилья и извлеки количество человек, которые будут проживать.

Текст заявки: {text}

Верни ТОЛЬКО ЧИСЛО (целое число), соответствующее количеству проживающих.
Если количество не указано явно, постарайся определить его по контексту.

Количество человек:'''
)

# Создание цепочки
chain = basic_prompt | llm | StrOutputParser()

# Тестирование
test_texts = [
    'Сниму недорогое жилье в лазаревском на 6 чел 3 взр и 3 дет 30.07-09.08',
    'Здравствуйте.Снимем жилье с 12.07 по 17.07. 2 взрослых и 1 ребенок 6 лет,со всеми удобствами недалеко от моря. Приезд в час ночи.',
    'Добрый вечер! Ищем жилье на 2 человек, с 3 сентября на 7 дней.',
    'Ищем жильё на 4 взрослых с 31.08.18 по 07.08.18, желательно центр',
    'Добрый день! Срочно ищем жилье с 13.06 по 20.06 нас 3  человека(все взрослые). Желательно с удобствами в номере. У Вас есть какие-нибудь варианты? Спасибо',
    'Ищем жилье в самом Лазаревском для двоих, 2-местный номер с удобствами, с 7-8 сентября не менее чем на неделю, 700 р. за номер',
    'Ищем жильё 2 взрослых и ребёнок 6 лет. С 8 июля по 21 июля',
    'Здравствуйте,ищем жильё 😊. Лазоревское с 29.06. По 25.07. 4чел. ( 2 взрослых и 2 детей) сумма не более 65000 р. Квартира 2-х ком.',
    'Интересует жилье на одного человека с 08.07. до 15.07. Не далеко от пляжа 🤗',
    'Добрый день. Сниму жильё на 2 недели. Конец августа-начала сентября. Два взрослых человека. По адекватной цене в приличном месте. Пишите в лс.',
    'Здравствуйте! Ищем жильё на августа числа с 6-7 на недельку. 2 взрослых и 2 ребенка 6 и 9 лет! Желательно номер в пределах 2000 или 500р с человека',
    'Сниму жилье в июне у моря 2 взрослых до 1500 в сутки',
    'Добрый день! Интересует жилье с 10-20 мая. 2 взрослых 1 ребенок. В Лазаревском',
    'Здравствуйте, интересует жилье п.Лазаревское 2 взр + 2 реб. 6 и 8 лет с 08.08 по 21.08? Недорого! Рядом с морем!',
    'Снимем жилье в Лазаревском с 30.07 по 09.08.15 г. Нас 3 взрослых, 2 детей 5 лет и 3,9 лет.'
]

for text in test_texts:
    result = chain.invoke({'text': text})
    print(f'Текст: {text}')
    print(f'Результат: {result}')
    print('---')

Текст: Сниму недорогое жилье в лазаревском на 6 чел 3 взр и 3 дет 30.07-09.08
Результат: 6
---
Текст: Здравствуйте.Снимем жилье с 12.07 по 17.07. 2 взрослых и 1 ребенок 6 лет,со всеми удобствами недалеко от моря. Приезд в час ночи.
Результат: 3
---
Текст: Добрый вечер! Ищем жилье на 2 человек, с 3 сентября на 7 дней.
Результат: 2
---
Текст: Ищем жильё на 4 взрослых с 31.08.18 по 07.08.18, желательно центр
Результат: 4
---
Текст: Добрый день! Срочно ищем жилье с 13.06 по 20.06 нас 3  человека(все взрослые). Желательно с удобствами в номере. У Вас есть какие-нибудь варианты? Спасибо
Результат: 3
---
Текст: Ищем жилье в самом Лазаревском для двоих, 2-местный номер с удобствами, с 7-8 сентября не менее чем на неделю, 700 р. за номер
Результат: 2
---
Текст: Ищем жильё 2 взрослых и ребёнок 6 лет. С 8 июля по 21 июля
Результат: 3
---
Текст: Здравствуйте,ищем жильё 😊. Лазоревское с 29.06. По 25.07. 4чел. ( 2 взрослых и 2 детей) сумма не более 65000 р. Квартира 2-х ком.
Результат: 4
---
Текст: 

In [18]:
import pandas as pd

# Загрузка данных из csv в датафрейм df
df = pd.read_csv('rental_09.csv', sep=';')

# Просмотр первых 5 строк
df.head()

,amount,text
0,6,Сниму недорогое жилье в лазаревском на 6 чел 3...
1,3,Здравствуйте.Снимем жилье с 12.07 по 17.07\r\n...
2,2,"Добрый вечер! Ищем жилье на 2 человек, с 3 сен..."
3,4,Ищем жильё на 4 взрослых с 31.08.18 по 07.08.1...
4,3,Добрый день! Срочно ищем жилье с 13.06 по 20.0...


In [19]:
results = []

for _, row in df.iterrows():
    text = row['text']
    try:
        result = chain.invoke({'text': text})
        results.append(result)
    except Exception as e:
        results.append(f'ERROR: {e}')

df['result'] = results
df.to_csv('rental_with_results.csv', index=False, encoding='utf-8-sig')

In [20]:
print(df['result'].dtypes)

str


In [21]:
correct = 0
for _, row in df.iterrows():
    if pd.to_numeric(row['result'], errors='coerce') == row['amount']:
        correct += 1

total = len(df)
accuracy = correct / total

print(f'Ошибок: {total - correct}')
print(f'Точность: {accuracy:.1%}')

Ошибок: 0
Точность: 100.0%
